# LAB | Feature Engineering

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


**Check the shape of your data**

In [3]:
spaceship.shape

(8693, 14)

**Check for data types**

In [4]:
spaceship.dtypes

PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object

**Check for missing values**

In [5]:
print(spaceship.isna().sum())
print(f"\nShare of rows with at least one missing value: {spaceship.isna().any(axis=1).mean():.1%}")

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

Share of rows with at least one missing value: 24.0%


There are multiple strategies to handle missing data

- Removing all rows or all columns containing missing data.
- Filling all missing values with a value (mean in continouos or mode in categorical for example).
- Filling all missing values with an algorithm.

For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

In [6]:
spaceship = spaceship.dropna().reset_index(drop=True)
print(spaceship.shape)
spaceship.isna().sum().sum()

(6606, 14)


np.int64(0)

- **Cabin** is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

In [7]:
# Cabin format is deck/num/side -> keep only the deck letter
spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]
print(sorted(spaceship["Cabin"].unique()))
spaceship["Cabin"].value_counts()

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'T']


Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64

- Drop PassengerId and Name

In [8]:
spaceship = spaceship.drop(columns=["PassengerId", "Name"])
spaceship.columns.tolist()

['HomePlanet',
 'CryoSleep',
 'Cabin',
 'Destination',
 'Age',
 'VIP',
 'RoomService',
 'FoodCourt',
 'ShoppingMall',
 'Spa',
 'VRDeck',
 'Transported']

- For non-numerical columns, do dummies.

In [9]:
# Separate the target from the features, then dummy-encode the non-numerical features
y = spaceship["Transported"].astype(int)
X = spaceship.drop(columns=["Transported"])

non_numeric = X.select_dtypes(exclude="number").columns.tolist()
print("Columns to encode:", non_numeric)

X = pd.get_dummies(X, columns=non_numeric, drop_first=True, dtype=int)
X.head()

Columns to encode: ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP']


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,1,0,0,1,0,0,0,0,0,0,0,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,0,0,0,0,0,0,0,1,0,0,0,1,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,1,0,0,0,0,0,0,0,0,0,0,1,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,1,0,0,0,0,0,0,0,0,0,0,1,0
4,16.0,303.0,70.0,151.0,565.0,2.0,0,0,0,0,0,0,0,1,0,0,0,1,0


**Perform Train Test Split**

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

(5284, 19) (1322, 19)


**Model Selection**

In this exercise we will be using **KNN** as our predictive model.

In [11]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print("Model trained:", knn)

Model trained: KNeighborsClassifier()


- Evaluate your model's performance. Comment it

In [12]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_train = knn.predict(X_train)
y_pred_test = knn.predict(X_test)

print(f"Train accuracy: {accuracy_score(y_train, y_pred_train):.3f}")
print(f"Test accuracy:  {accuracy_score(y_test, y_pred_test):.3f}")
print(f"Baseline (majority class): {max(y_test.mean(), 1 - y_test.mean()):.3f}\n")
print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))

Train accuracy: 0.830
Test accuracy:  0.774
Baseline (majority class): 0.504

[[500 156]
 [143 523]]
              precision    recall  f1-score   support

           0       0.78      0.76      0.77       656
           1       0.77      0.79      0.78       666

    accuracy                           0.77      1322
   macro avg       0.77      0.77      0.77      1322
weighted avg       0.77      0.77      0.77      1322



**Comments on the model's performance**

- The KNN model (k=5) reaches about **77% accuracy on the test set**, versus about 50% for a naive majority-class baseline (the target is almost perfectly balanced), so it clearly learns useful signal.
- Precision and recall are well balanced across both classes (≈0.77–0.79), so the model does not favour one class over the other.
- Train accuracy (≈83%) is higher than test accuracy (≈77%): a mild overfitting, which is typical for KNN.
- The numeric spending features (RoomService, FoodCourt, Spa, VRDeck...) have a much larger scale than the dummy variables and Age. Since KNN is distance-based, these features dominate the distance computation.
- Possible improvements: scale the features (StandardScaler / MinMaxScaler), tune `n_neighbors` with cross-validation, and test other models (logistic regression, random forest, XGBoost).